# Question and Answering

Three systems shaped modern QA.

* Extractive found spans.
* Retrieval-augmented groudned them in documents.
* Generative produced answers.

## Problem Definition

Tree architectures have dominated QA over the last decade.

### Extractive QA

Given a question and a passage that is known to contain the answer, find the **start and end** indices of the answer span in passage.

### Open-domain QA

Retrieve the relevant passage first, then extract or generate an answer.

### Generative/Closed-book QA

A large language model answers from its parametric memory.

### Hybird -- trend in 2026

Retrieve the best few passages, then prompt a generative model to answer grounded in those passages. That is RAG.

## Basic Concept

### Extractive.

Encode question and passage together with a transformer. Train two heads that predict start and end token indices of the answer. Loss is cross entropy over valid positions.

### Retrieval-augmented (RAG).

First, a retriver finds the top-k passages from a corpus.
Second, a reader produces the answer using those passages.

### Generative

A decoder-only LLM answers from learned weights. No retrieval step.
Excellent on common knowledge, catastrophic on rare or recent facts.

# Build your Own

## Extrative QA with a pretrained model

In [8]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

model_name = "deepset/roberta-base-squad2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForQuestionAnswering.from_pretrained(model_name)

passage = (
    "Apple Inc. released the first iPhone on June 29, 2007. "
    "The device was announced by Steve Jobs at Macword in January 2007. "
)

question = "When was the first iPhone released?"

inputs = tokenizer(question, passage, return_tensors="pt", truncation=True)
outputs = model(**inputs)
print(outputs)

answer_start = outputs.start_logits.argmax()
answer_end = outputs.end_logits.argmax() + 1
answer = tokenizer.decode(inputs["input_ids"][0][answer_start:answer_end])

print(answer)

def qa(question, context):
    inputs = tokenizer(question, context, return_tensors="pt", truncation=True)
    outputs = model(**inputs)
    #print(outputs)

    answer_start = outputs.start_logits.argmax()
    answer_end = outputs.end_logits.argmax() + 1

    return tokenizer.decode(inputs["input_ids"][0][answer_start:answer_end])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

QuestionAnsweringModelOutput(loss=None, start_logits=tensor([[ 0.8628, -8.4593, -8.7280, -8.6168, -8.0570, -8.3445, -8.2433, -8.9029,
         -9.2025, -9.0339, -1.3949, -5.2076, -7.5652, -1.8156, -1.5804, -1.9678,
         -3.5092, -0.1740,  7.5982, -1.0574, -3.2712,  1.6070, -3.4183, -4.0338,
         -6.0000, -6.9231, -4.7176, -7.2580, -4.7230, -6.0294, -6.0451, -3.0784,
         -6.2895, -4.6626,  0.6973,  0.2648, -3.4183, -3.5230, -9.0031]],
       grad_fn=<CloneBackward0>), end_logits=tensor([[ 0.9059, -7.4236, -7.6322, -8.1012, -8.1370, -6.9866, -7.1908, -5.8118,
         -6.8315, -5.4107, -4.2765, -6.5135, -5.0902, -5.6808, -6.2463, -5.2038,
         -3.5374, -4.0359,  0.6747,  0.7086, -0.6270,  8.0333,  2.1787, -7.2637,
         -6.2842, -7.7741, -6.0427, -7.8499, -7.4672, -4.3260, -7.9627, -7.1027,
         -4.2695, -7.5089, -2.2220,  2.4110,  2.1787, -2.2874, -7.8699]],
       grad_fn=<CloneBackward0>), hidden_states=None, attentions=None)
 June 29, 2007


## Retrieval-augmented pipeline

In [10]:
from sentence_transformers import SentenceTransformer
import numpy as np

encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

corpus = [
    "Apple Inc. released the first iPhone on June 29, 2007.",
    "Macworld 2007 featured the iPhone announcement by Steve Jobs.",
    "Android launched in 2008 as Google's mobile operating system.",
    "The first iPod was released in 2001.",
]

corpus_embeddings = encoder.encode(corpus, normalize_embeddings=True)

def retrieve(question, top_k=2):
    q_emb = encoder.encode([question], normalize_embeddings=True)
    sims = (corpus_embeddings @ q_emb.T).squeeze()
    order = np.argsort(-sims)[:top_k]
    return [corpus[i] for i in order]

def answer(question):
    passages = retrieve(question, top_k=2)
    combined = "".join(passages)
    return qa(question=question, context=combined)

print(answer("When was the first iPhone released?"))
print(answer("When was the first iPod released?"))

def rag_generate(question, llm):
    passages = retrieve(question, top_k=3)
    prompt = f"""Context:
    {chr(10).join('-' + p for p in passages)}

    Question: {question}

    Answer using only the context above. If the context does not contain the answer, say "I don't know"."""
    return llm(prompt)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

 June 29, 2007
 2001


## Evaluation

For production QA:
* Answer accuracy.       LLM-judged or human-judged
* Citation accuracy.     Does the cited passage actually support the answer.
* Refusal calibration.   When the answer is not in the retrieved passages, does the system correctly say "I don't know" ?